In [3]:
import pandas as pd
df = pd.read_csv("/content/lstm_text_dataset_large.csv")
df.head(12)

,text
0,I love machine learning 0
1,machine learning is fun 0
2,deep learning is powerful 0
3,I enjoy learning new things 0
4,artificial intelligence is the future 0
5,data science is very interesting 0
6,I like building AI models 0
7,learning Python is useful 0
8,neural networks are amazing 0
9,I love coding in Python 0


In [5]:
# Convert text column into list
text_data = df['text'].astype(str).tolist()

print(text_data[:5])

['I love machine learning 0', 'machine learning is fun 0', 'deep learning is powerful 0', 'I enjoy learning new things 0', 'artificial intelligence is the future 0']


In [6]:
import re

cleaned_text = []

for line in text_data:
    line = line.lower()  # lowercase
    line = re.sub(r'[^a-zA-Z ]', '', line)  # remove symbols & numbers
    cleaned_text.append(line)

print(cleaned_text[:5])

['i love machine learning ', 'machine learning is fun ', 'deep learning is powerful ', 'i enjoy learning new things ', 'artificial intelligence is the future ']


In [7]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer()
tokenizer.fit_on_texts(cleaned_text)

total_words = len(tokenizer.word_index) + 1
print(total_words)

134


In [9]:
input_sequences = []

for line in cleaned_text:
    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

# Print results
print("Total sequences generated:", len(input_sequences))
print("\nSample sequences:")
print(input_sequences[:10])

Total sequences generated: 1152

Sample sequences:
[[2, 20], [2, 20, 21], [2, 20, 21, 3], [21, 3], [21, 3, 1], [21, 3, 1, 47], [12, 3], [12, 3, 1], [12, 3, 1, 48], [2, 22]]


In [10]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Find max sequence length
max_sequence_len = max(len(seq) for seq in input_sequences)

# Apply padding
input_sequences = pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')

# Print outputs
print("Maximum sequence length:", max_sequence_len)
print("\nPadded sequences (first 5):")
print(input_sequences[:5])

Maximum sequence length: 7

Padded sequences (first 5):
[[ 0  0  0  0  0  2 20]
 [ 0  0  0  0  2 20 21]
 [ 0  0  0  2 20 21  3]
 [ 0  0  0  0  0 21  3]
 [ 0  0  0  0 21  3  1]]


In [11]:
import numpy as np

input_sequences = np.array(input_sequences)

X = input_sequences[:, :-1]
y = input_sequences[:, -1]
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1152, 6)
y shape: (1152,)


In [12]:
from tensorflow.keras.utils import to_categorical

y = to_categorical(y, num_classes=total_words)

print("After encoding y shape:", y.shape)

After encoding y shape: (1152, 134)


In [13]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

model = Sequential()
model.add(Embedding(total_words, 100, input_length=X.shape[1]))
model.add(LSTM(150))
model.add(Dense(total_words, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [14]:
history = model.fit(X, y, epochs=50, verbose=1)

Epoch 1/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.0799 - loss: 4.7594
Epoch 2/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.0833 - loss: 4.4694
Epoch 3/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.1319 - loss: 4.1767
Epoch 4/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.2101 - loss: 3.5468
Epoch 5/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.2908 - loss: 2.8659
Epoch 6/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.4523 - loss: 2.2543
Epoch 7/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.6424 - loss: 1.7276
Epoch 8/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7483 - loss: 1.3389
Epoch 9/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7986 - loss: 1.0487
Epoch 10/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.8316 - loss: 0.8307
Epoch 11/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.8368 - loss: 0.6883
Epoch 12/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy:

In [15]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_next_word(model, tokenizer, text, max_sequence_len):

    text = text.lower()

    token_list = tokenizer.texts_to_sequences([text])[0]

    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')

    predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)

    for word, index in tokenizer.word_index.items():
        if index == predicted:
            return word

In [20]:
while True:
    text = input("Enter your sentence (or type 'end'): ")

    if text.lower() == "end":
        print("Program ended.")
        break

    next_word = predict_next_word(model, tokenizer, text, max_sequence_len)

    print("Predicted next word:", next_word)

Enter your sentence (or type 'end'): i am so much in
Predicted next word: deep
Enter your sentence (or type 'end'): while i stay here you
Predicted next word: exploring
Enter your sentence (or type 'end'): yes and
Predicted next word: is
Enter your sentence (or type 'end'): what do you
Predicted next word: model
Enter your sentence (or type 'end'): end
Program ended.


In [17]:
print(predict_next_word(model, tokenizer, "machine learning", max_sequence_len))
print(predict_next_word(model, tokenizer, "deep learning", max_sequence_len))
print(predict_next_word(model, tokenizer, "i am", max_sequence_len))

is
uses
working


In [39]:
model.save("lstm_autocomplete_model.h5")

In [40]:
from google.colab import files
files.download("lstm_autocomplete_model.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [41]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [42]:
files.download("tokenizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [43]:
print(max_sequence_len)

7
